## Pretraining

### Load all allowed libraries

In [53]:
import numpy as np
from PIL import Image
import pandas as pd
import sklearn
import scipy
import seaborn as sns
import torch
import torchinfo
import torchvision
from tqdm import tqdm

import matplotlib.pyplot as plt

### Define data transfromation function, and data augmentation

In [54]:
from torchvision.transforms import v2

# Training data transforms and augmentation
training_tfms_and_agmt = v2.Compose([
    v2.ToImage(), # this turns it into a tensor

    v2.RandomRotation(degrees=10), # for realistic pet location variations 
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1)), # for better position accuracy
    v2.RandomHorizontalFlip(p=0.5), # it is still the same cat if flip like this but double data

    v2.RandomResizedCrop((224, 224), scale=(0.8, 1.0), antialias=True), # 224 * 224 seems to be what everyone doing
    v2.ColorJitter(brightness=0.3, contrast=0.2, saturation=0.2, hue=0.05), # this is for pet photographic varience

    v2.ToDtype(torch.float32, scale=True), # apply the min-max normalization and scale is for 0-255 to num between 0 and 1
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # apply the standardization for the z-scores things
])

valutation_tfms = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # this normalization values are for now, can be better
])

### Load dataset and create custom dataset

In [55]:
from torch.utils.data import Dataset, random_split
from torchvision import datasets

class TransformWrapper(Dataset):
    def __init__(self, subset, transform=None):
        super().__init__() # inherit the Dataset init method just in case even though I think there is nothing
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

raw_data = datasets.OxfordIIITPet(root="/shared/storage/cs/studentscratch/cqh514", split='trainval', download=True)

### Randomly split but make the split constant accross different runes, in a 97.5:2.5 training, valuation configuration

In [56]:
# do the dataset splitting
dataset_size = len(raw_data)
num_training = int(0.975 * dataset_size)
num_valuation = dataset_size - num_training

# define the manual seed so it no change after training
generator = torch.Generator().manual_seed(67)
raw_training, raw_valuation = random_split(raw_data, [num_training, num_valuation], generator=generator)

### Inject the transforms to the raw subset that is created with the split we just made

In [57]:
# Wrap the raw subsets to inject the transforms on the fly
training_dataset = TransformWrapper(raw_training, transform=training_tfms_and_agmt)
valuation_dataset = TransformWrapper(raw_valuation, transform=valutation_tfms)

### Preparing data with dataloaders, don't shuffle the valuation then no waste cpu

In [58]:
from torch.utils.data import DataLoader
training_loader = DataLoader(training_dataset, batch_size=32, shuffle=True, num_workers=10)
valuation_loader = DataLoader(valuation_dataset, batch_size=32, shuffle=False, num_workers=10)

### Define the first 1080ti in the server for training

# Training

### Stuff to change to run on different GPUs

In [59]:
model_save_path = "/home/userfs/c/cqh514/Documents/Oxford-IIIT-Pet-Classifier/pet_classifier_weights.pth"
device = torch.device("cuda:5") # cuda:1 is gpu 1, cuda:2 is gpu 2, etc

### Define the neural network by subclassing the nn.Module

In [60]:
import torch.nn as nn

class OxfordPetClassifierCNN(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        
        # Convolutional Layers, adding more then 5 seems to decrease accuracy
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        
        # Only one linear Layer for now so it at least works
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        output = self.classifier(x) # logits is the name for the numbers outputed before softmax turns it into probabilities
        return output

model = OxfordPetClassifierCNN().to(device)
print(model)

OxfordPetClassifierCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_runnin

### Set loss function and its optimizer

In [61]:
# use the cross entropy loss with label smoothing, instead of like 1 or 0 for classification, make it less definitive
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Adam optimizer seems to be the best and industry standard
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

### Run the training loop

In [62]:
import time

EPOCHS = 30
best_val_acc = 0.0

# Tell the scheduler exactly how many total steps there are
total_steps = len(training_loader) * EPOCHS

# The one cycle Scheduler, basicly peak learning rate at epoch 15 and slow down by epoch 30, so it will be best for my 30 epoch limit
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3, # peak learning rate
    total_steps=total_steps, # this is basicly 30
    pct_start=0.3, # 30% is warm up 
    div_factor=25.0, # starting learning rate
    final_div_factor=1000.0 # ending learning rate
)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    start_time = time.time() # for seeing how long a epoch takes and might just use another gpu
    
    # training starts here
    model.train() # this basicly activates the dropout but i removed it now just normalize based on current batch's mean/varience
    running_loss = 0.0 # rset stuff after each epoch
    correct_train = 0
    total_train = 0
    
    for inputs, labels in tqdm(training_loader, desc="Training", leave=False): # the tqdm is the progress bar thing
        inputs, labels = inputs.to(device), labels.to(device) # move data to the vram so it faster
        
        optimizer.zero_grad() # clear gradient from last batch
        outputs = model(inputs) # forward the data
        loss = criterion(outputs, labels) # run loss function
        loss.backward() # back probagation
        
        optimizer.step() # update the weights 
        
        scheduler.step() # update the learning rate for the one cycle lr
        
        running_loss += loss.item() * inputs.size(0) # loss per epoch
        _, predicted = torch.max(outputs, 1) # get the prediciton no need the score
        total_train += labels.size(0) # get total images
        correct_train += (predicted == labels).sum().item() # count correct ones
        
    train_acc = correct_train / total_train # the training accuracy
    
    # validation starts here
    model.eval() # validation mode stop the droputs + batchnorm uses running averages
    correct_val = 0
    total_val = 0
    
    # no need gradient calculations for testing
    with torch.no_grad():
        for inputs, labels in tqdm(valuation_loader, desc="Validating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device) # move to vram again
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1) # basicly duplicated code from the training phase
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_acc = correct_val / total_val

    current_lr = optimizer.param_groups[0]['lr'] # current learning rate
    
    print(f"LR: {current_lr:.5f} | Train Acc: {train_acc * 100:.2f}% | Val Acc: {val_acc * 100:.2f}% | Time: {time.time() - start_time:.0f}s")
    
    # save the model only if the validation accuracy better
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), model_save_path)
        print(f"🌟 Saved new best model with Val Acc: {val_acc * 100:.2f}%")

print(f"\nTraining done! Best validation accuracy: {best_val_acc * 100:.2f}%")

Epoch 1/30
---------------


Training:   0%|                                                                                                                                  | 0/113 [00:00<?, ?it/s]

LR: 0.00021 | Train Acc: 7.50% | Val Acc: 11.96% | Time: 22s
🌟 Saved new best model with Val Acc: 11.96%
Epoch 2/30
---------------


LR: 0.00046 | Train Acc: 11.40% | Val Acc: 7.61% | Time: 35s
Epoch 3/30
---------------


LR: 0.00084 | Train Acc: 11.98% | Val Acc: 15.22% | Time: 26s
🌟 Saved new best model with Val Acc: 15.22%
Epoch 4/30
---------------


...

LR: 0.00026 | Train Acc: 77.42% | Val Acc: 60.87% | Time: 34s
🌟 Saved new best model with Val Acc: 60.87%
Epoch 27/30
---------------


LR: 0.00015 | Train Acc: 79.40% | Val Acc: 60.87% | Time: 37s
Epoch 28/30
---------------


LR: 0.00007 | Train Acc: 80.88% | Val Acc: 63.04% | Time: 34s
🌟 Saved new best model with Val Acc: 63.04%
Epoch 29/30
---------------


LR: 0.00002 | Train Acc: 82.05% | Val Acc: 63.04% | Time: 31s
Epoch 30/30
---------------


LR: 0.00000 | Train Acc: 82.86% | Val Acc: 65.22% | Time: 43s
🌟 Saved new best model with Val Acc: 65.22%

Training done! Best validation accuracy: 65.22%


### Testing transform (same as the valuation transform)

In [63]:
from torchvision.transforms import v2


test_tfms = v2.Compose([ # this needs to be same as the valuation transform
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [64]:
raw_test_data = datasets.OxfordIIITPet(root="/shared/storage/cs/studentscratch/cqh514", split='test', download=True)
test_dataset = TransformWrapper(raw_test_data, transform=test_tfms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=10)

In [65]:
model = OxfordPetClassifierCNN(num_classes=37).to(device)
model.load_state_dict(torch.load(model_save_path))

# set to evaulation mode
model.eval()

correct_test = 0
total_test = 0

print("Starting evaluation on the test dataset...")

# disable gradient calculations for testing, basicly same code as the validation phase in the training loop
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_acc = correct_test / total_test
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

Starting evaluation on the test dataset...


Testing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 115/115 [00:10<00:00, 11.08it/s]



Final Test Accuracy: 58.03%
